# WFCRL Synthetic Graph PFN Sandbox
Synthetic wake-graph dataset + baselines + PFN-style in-context graph regressor scaffold.


## 0) Install dependencies
Skip if already installed. `torch-geometric` install varies by OS/CUDA.


In [1]:
# If running locally, uncomment as needed.
# !pip install -U numpy torch tabpfn
# For torch-geometric, follow:
# https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html

import numpy as np


## 1) Synthetic wake graph generator

In [2]:
from __future__ import annotations
import math
import numpy as np
from dataclasses import dataclass
from typing import Tuple, List

@dataclass
class GraphSample:
    x: np.ndarray              # (N, F)
    edge_index: np.ndarray     # (2, E)
    edge_attr: np.ndarray      # (E, A)
    y_total: float
    y_node: np.ndarray         # (N,)

def _angle_wrap_deg(a: float) -> float:
    return (a + 360.0) % 360.0

def _angle_diff_deg(a: float, b: float) -> float:
    d = (a - b + 180.0) % 360.0 - 180.0
    return abs(d)

def _bearing_deg(dx: float, dy: float) -> float:
    # 0 deg = +y (north), 90 deg = +x (east)
    return _angle_wrap_deg(math.degrees(math.atan2(dx, dy)))

def build_wake_graph(
    xy: np.ndarray,
    wind_dir_deg: float,
    dist_thresh: float = 1200.0,
    cone_deg: float = 45.0,
) -> Tuple[np.ndarray, np.ndarray]:
    n = xy.shape[0]
    src, dst, eattr = [], [], []
    downwind = _angle_wrap_deg(wind_dir_deg + 180.0)

    for i in range(n):
        xi, yi = xy[i]
        for j in range(n):
            if i == j:
                continue
            xj, yj = xy[j]
            dx, dy = float(xj - xi), float(yj - yi)
            d = math.hypot(dx, dy)
            if d > dist_thresh:
                continue
            b = _bearing_deg(dx, dy)
            if _angle_diff_deg(b, downwind) <= cone_deg:
                src.append(i)
                dst.append(j)
                eattr.append([d, _angle_diff_deg(b, downwind), d / dist_thresh])

    edge_index = np.asarray([src, dst], dtype=np.int64)
    edge_attr  = np.asarray(eattr, dtype=np.float32) if len(eattr) else np.zeros((0, 3), np.float32)
    return edge_index, edge_attr

def jensen_like_power(
    xy: np.ndarray,
    yaw_deg: np.ndarray,
    wind_speed: float,
    wind_dir_deg: float,
    rotor_diam: float = 126.0,
    ct: float = 0.8,
    k_wake: float = 0.05,
    base_eff: float = 1.0,
) -> np.ndarray:
    n = xy.shape[0]
    ws = wind_speed
    wd = wind_dir_deg
    downwind = _angle_wrap_deg(wd + 180.0)

    th = math.radians(downwind)
    ux, uy = math.sin(th), math.cos(th)

    deficits2 = np.zeros(n, dtype=np.float64)

    yaw_rad = np.deg2rad(yaw_deg)
    yaw_gain = np.cos(yaw_rad) ** 1.88

    for i in range(n):
        xi, yi = xy[i]
        for j in range(n):
            if i == j:
                continue
            xj, yj = xy[j]
            dx, dy = (xj - xi), (yj - yi)

            s = dx * ux + dy * uy
            if s <= 0:
                continue

            cwx = dx - s * ux
            cwy = dy - s * uy
            r = math.hypot(cwx, cwy)

            R = rotor_diam / 2.0 + k_wake * s
            if r > R:
                continue

            yaw_factor = (math.cos(yaw_rad[i]) ** 1.2)
            denom = (1.0 + 2.0 * k_wake * s / rotor_diam) ** 2
            d_ij = (1.0 - math.sqrt(1.0 - ct * yaw_factor)) / denom
            deficits2[j] += d_ij ** 2

    ws_eff = ws * (1.0 - np.sqrt(deficits2))
    ws_eff = np.clip(ws_eff, 0.0, None)

    p = base_eff * (ws_eff ** 3) * yaw_gain
    return p.astype(np.float32)

def sample_synthetic_graph(
    n_turbines: int = 9,
    layout: str = "random",
    area: float = 2000.0,
    wind_speed_range: Tuple[float, float] = (6.0, 12.0),
    wind_dir_range: Tuple[float, float] = (0.0, 360.0),
    yaw_range: Tuple[float, float] = (-25.0, 25.0),
    rng: np.random.Generator | None = None,
    ct: float = 0.8,
    k_wake: float = 0.05,
) -> GraphSample:
    rng = rng or np.random.default_rng()

    if layout == "grid":
        side = int(math.ceil(math.sqrt(n_turbines)))
        xs = np.linspace(0, area, side)
        ys = np.linspace(0, area, side)
        xy = np.array([(x, y) for x in xs for y in ys], dtype=np.float32)[:n_turbines]
    elif layout == "row":
        xy = np.stack([np.linspace(0, area, n_turbines), np.full(n_turbines, area * 0.5)], axis=1).astype(np.float32)
    else:
        xy = rng.uniform(0, area, size=(n_turbines, 2)).astype(np.float32)

    wind_speed = float(rng.uniform(*wind_speed_range))
    wind_dir = float(rng.uniform(*wind_dir_range))
    yaw = rng.uniform(*yaw_range, size=(n_turbines,)).astype(np.float32)

    edge_index, edge_attr = build_wake_graph(xy, wind_dir)

    x = np.column_stack([
        yaw,
        np.full(n_turbines, wind_speed, dtype=np.float32),
        np.full(n_turbines, wind_dir, dtype=np.float32),
        xy[:, 0],
        xy[:, 1],
    ]).astype(np.float32)

    p_node = jensen_like_power(xy, yaw, wind_speed, wind_dir, ct=ct, k_wake=k_wake)
    y_total = float(np.sum(p_node))
    return GraphSample(x=x, edge_index=edge_index, edge_attr=edge_attr, y_total=y_total, y_node=p_node)

def make_dataset(num_samples: int = 5000, n_turbines: int = 9, seed: int = 0) -> List[GraphSample]:
    rng = np.random.default_rng(seed)
    return [sample_synthetic_graph(n_turbines=n_turbines, rng=rng) for _ in range(num_samples)]

s = sample_synthetic_graph(n_turbines=9, layout="random", rng=np.random.default_rng(0))
print("x:", s.x.shape, "edge_index:", s.edge_index.shape, "edge_attr:", s.edge_attr.shape, "y_total:", s.y_total)
print("num edges:", s.edge_index.shape[1])


x: (9, 5) edge_index: (2, 13) edge_attr: (13, 3) y_total: 3145.595703125
num edges: 13


## 2) Graph baseline (GNN) on synthetic data

In [3]:
import torch
import torch.nn as nn

try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GATv2Conv, global_mean_pool
except Exception as e:
    raise RuntimeError(
        "torch_geometric not available. Install PyTorch Geometric first.\n"
        "See: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html\n"
        f"Original error: {e}"
    )

def to_data(s: GraphSample) -> Data:
    return Data(
        x=torch.tensor(s.x, dtype=torch.float32),
        edge_index=torch.tensor(s.edge_index, dtype=torch.long),
        edge_attr=torch.tensor(s.edge_attr, dtype=torch.float32),
        y=torch.tensor([s.y_total], dtype=torch.float32),
    )

class GNNReg(nn.Module):
    def __init__(self, in_dim=5, hid=64):
        super().__init__()
        self.c1 = GATv2Conv(in_dim, hid, heads=2, concat=False, edge_dim=3)
        self.c2 = GATv2Conv(hid, hid, heads=2, concat=False, edge_dim=3)
        self.lin = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))

    def forward(self, data: Data):
        h = self.c1(data.x, data.edge_index, data.edge_attr).relu()
        h = self.c2(h, data.edge_index, data.edge_attr).relu()
        g = global_mean_pool(h, data.batch)
        return self.lin(g).squeeze(-1)

ds = [to_data(s) for s in make_dataset(num_samples=3000, n_turbines=9, seed=0)]
tr, te = ds[:2600], ds[2600:]

tr_loader = DataLoader(tr, batch_size=64, shuffle=True)
te_loader = DataLoader(te, batch_size=256)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNNReg().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

for ep in range(15):
    model.train()
    for b in tr_loader:
        b = b.to(device)
        pred = model(b)
        loss = loss_fn(pred, b.y.view(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        se = 0.0
        n = 0
        for b in te_loader:
            b = b.to(device)
            pred = model(b)
            se += torch.sum((pred - b.y.view(-1)) ** 2).item()
            n += b.num_graphs
        mse = se / n
    print(f"epoch={ep:02d} test_mse={mse:.4f} device={device}")


epoch=00 test_mse=40998968.3200 device=cpu
epoch=01 test_mse=30977580.8000 device=cpu
epoch=02 test_mse=12408327.0400 device=cpu
epoch=03 test_mse=11216598.0800 device=cpu
epoch=04 test_mse=11176332.8000 device=cpu
epoch=05 test_mse=11142080.3200 device=cpu
epoch=06 test_mse=11096705.9200 device=cpu
epoch=07 test_mse=11074498.2400 device=cpu
epoch=08 test_mse=11039627.5200 device=cpu
epoch=09 test_mse=11008737.9200 device=cpu
epoch=10 test_mse=10981305.6000 device=cpu
epoch=11 test_mse=10994659.2000 device=cpu
epoch=12 test_mse=10936585.2800 device=cpu
epoch=13 test_mse=10907548.1600 device=cpu
epoch=14 test_mse=10892224.0000 device=cpu


## 3) Tabular PFN baseline on flattened node features

In [4]:
from tabpfn import TabPFNRegressor

rng = np.random.default_rng(0)
samples = [sample_synthetic_graph(n_turbines=9, rng=rng) for _ in range(1200)]
X = np.stack([s.x.reshape(-1) for s in samples], axis=0).astype(np.float32)
y = np.asarray([s.y_total for s in samples], dtype=np.float32)

idx = np.arange(len(samples))
rng.shuffle(idx)
tr_idx, te_idx = idx[:1000], idx[1000:]
X_tr, y_tr = X[tr_idx], y[tr_idx]
X_te, y_te = X[te_idx], y[te_idx]

model = TabPFNRegressor()
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
mse = float(np.mean((pred - y_te) ** 2))
print("X_tr shape:", X_tr.shape, "X_te shape:", X_te.shape)
print("TabPFN test MSE:", mse)


X_tr shape: (1000, 45) X_te shape: (200, 45)
TabPFN test MSE: 501947.03125


## 4) Synthetic geometric equilibrium labels (best-response)

In [5]:
def best_response_equilibrium(
    xy: np.ndarray,
    wind_speed: float,
    wind_dir: float,
    iters: int = 30,
    yaw_grid: np.ndarray | None = None,
    ct: float = 0.8,
    k_wake: float = 0.05,
) -> np.ndarray:
    n = xy.shape[0]
    yaw = np.zeros(n, dtype=np.float32)
    yaw_grid = yaw_grid if yaw_grid is not None else np.linspace(-25, 25, 21).astype(np.float32)

    for _ in range(iters):
        changed = False
        for i in range(n):
            best_y = yaw[i]
            best_val = -1e18
            for y in yaw_grid:
                yaw_try = yaw.copy()
                yaw_try[i] = y
                p = jensen_like_power(xy, yaw_try, wind_speed, wind_dir, ct=ct, k_wake=k_wake)
                val = float(np.sum(p))
                if val > best_val:
                    best_val = val
                    best_y = y
            if best_y != yaw[i]:
                yaw[i] = best_y
                changed = True
        if not changed:
            break
    return yaw

rng = np.random.default_rng(1)
s = sample_synthetic_graph(n_turbines=9, layout="grid", rng=rng)
xy = s.x[:, 3:5].astype(np.float32)
wind_speed = float(s.x[0, 1])
wind_dir = float(s.x[0, 2])

yaw_eq = best_response_equilibrium(xy, wind_speed, wind_dir)
p_eq = jensen_like_power(xy, yaw_eq, wind_speed, wind_dir)
print("equilibrium yaw:", yaw_eq)
print("total power at eq:", float(np.sum(p_eq)))
print("total power at sample yaw:", s.y_total)


equilibrium yaw: [0. 0. 0. 0. 0. 0. 0. 0. 0.]
total power at eq: 6717.3486328125
total power at sample yaw: 6318.38623046875


## 5) PFN-style in-context graph regressor (minimal B=1 task batching)

In [6]:
import torch
import torch.nn as nn

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool

class GraphEncoder(nn.Module):
    def __init__(self, in_dim: int, hid: int = 64):
        super().__init__()
        self.g1 = GATv2Conv(in_dim, hid, heads=2, concat=False, edge_dim=3)
        self.g2 = GATv2Conv(hid, hid, heads=2, concat=False, edge_dim=3)
        self.lin = nn.Linear(hid, hid)

    def forward(self, batch_data: Batch):
        edge_index = batch_data.edge_index  # keep as int64
        edge_attr  = batch_data.edge_attr

        h = self.g1(batch_data.x, edge_index, edge_attr).relu()
        h = self.g2(h, edge_index, edge_attr).relu()
        g = global_mean_pool(h, batch_data.batch)
        return self.lin(g)

class CrossAttnRegressor(nn.Module):
    def __init__(self, emb: int = 64, heads: int = 4):
        super().__init__()
        self.q_proj = nn.Linear(emb, emb)
        self.k_proj = nn.Linear(emb + 1, emb)
        self.v_proj = nn.Linear(emb + 1, emb)
        self.attn = nn.MultiheadAttention(embed_dim=emb, num_heads=heads, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(emb, emb), nn.ReLU(), nn.Linear(emb, 1))

    def forward(self, q_emb, ctx_emb, ctx_y):
        q = self.q_proj(q_emb).unsqueeze(0)                       # (1, Q, E)
        k_in = torch.cat([ctx_emb, ctx_y], dim=-1).unsqueeze(0)   # (1, C, E+1)
        k = self.k_proj(k_in)
        v = self.v_proj(k_in)
        out, _ = self.attn(q, k, v)
        pred = self.mlp(out.squeeze(0)).squeeze(-1)               # (Q,)
        return pred

class GraphPFNInContext(nn.Module):
    def __init__(self, node_dim: int = 5, emb: int = 64):
        super().__init__()
        self.enc = GraphEncoder(node_dim, hid=emb)
        self.head = CrossAttnRegressor(emb=emb)

    def forward(self, ctx_batch: Batch, ctx_y: torch.Tensor, q_batch: Batch):
        ctx_emb = self.enc(ctx_batch)   # (C, E)
        q_emb = self.enc(q_batch)       # (Q, E)
        return self.head(q_emb, ctx_emb, ctx_y)

def to_pyg(s: GraphSample) -> Data:
    return Data(
        x=torch.tensor(s.x, dtype=torch.float32),
        edge_index=torch.tensor(s.edge_index, dtype=torch.long),
        edge_attr=torch.tensor(s.edge_attr, dtype=torch.float32),
    )

def sample_task_batch(C: int = 16, Q: int = 16, n_turbines: int = 9, seed: int = 0):
    rng = np.random.default_rng(seed)
    ct = float(rng.uniform(0.6, 0.9))
    k_wake = float(rng.uniform(0.02, 0.08))
    layout = rng.choice(["random", "grid", "row"])

    ctx = [sample_synthetic_graph(n_turbines=n_turbines, layout=layout, rng=rng, ct=ct, k_wake=k_wake) for _ in range(C)]
    qry = [sample_synthetic_graph(n_turbines=n_turbines, layout=layout, rng=rng, ct=ct, k_wake=k_wake) for _ in range(Q)]

    ctx_batch = Batch.from_data_list([to_pyg(s) for s in ctx])
    q_batch   = Batch.from_data_list([to_pyg(s) for s in qry])

    ctx_y = torch.tensor([[s.y_total] for s in ctx], dtype=torch.float32)
    q_y   = torch.tensor([s.y_total for s in qry], dtype=torch.float32)
    return ctx_batch, ctx_y, q_batch, q_y

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphPFNInContext(node_dim=5, emb=64).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

for step in range(100):
    ctx_batch, ctx_y, q_batch, q_y = sample_task_batch(C=16, Q=16, seed=step)
    ctx_batch = ctx_batch.to(device)
    q_batch = q_batch.to(device)
    ctx_y = ctx_y.to(device)
    q_y = q_y.to(device)

    pred = model(ctx_batch, ctx_y, q_batch)
    loss = loss_fn(pred, q_y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 10 == 0:
        print("step", step, "loss", float(loss.item()), "device", device)


step 0 loss 37398520.0 device cpu
step 10 loss 50728708.0 device cpu
step 20 loss 24737284.0 device cpu
step 30 loss 26269678.0 device cpu
step 40 loss 31458412.0 device cpu
step 50 loss 20179752.0 device cpu
step 60 loss 19670664.0 device cpu
step 70 loss 10226608.0 device cpu
step 80 loss 19284830.0 device cpu
step 90 loss 12416038.0 device cpu


## 6) WFCRL integration hook

In [7]:
def wfcrl_obs_to_pyg(obs: dict, env):
    raise NotImplementedError("Implement using your WFCRL obs_to_graph and map to torch_geometric.data.Data.")
